# Campus Assistant ISU (unofficial) — the Android app

This notebook turns ReggieBot into **an Android app for Google Play**, named **Campus Assistant ISU (unofficial)**.
The app is the chat screen. The answering happens on a **new Hugging Face Space**, so this PC can be off.

```
 Android phone                        new Hugging Face Space (ZeroGPU)               private HF dataset
┌────────────────────────┐  HTTPS   ┌──────────────────────────────────────────┐   ┌──────────────────┐
│ Campus Assistant app   │ ───────▶ │ android_space_app.py                     │──▶│ reports/         │
│  chat, source links,   │   ask    │   endpoints: ask · report · status       │   │ usage_log.csv    │
│  Report button,        │ ◀─────── │ space_app.py: Gemma 4, the search, data  │   └──────────────────┘
│  About + privacy       │ streamed │ reggiebot/: the same package             │
└────────────────────────┘          └──────────────────────────────────────────┘
```

| Piece | File | |
|---|---|---|
| The app's server | `android_space_app.py` | **new**: the Space edition under the new name, plus the `ask`, `report` and `status` endpoints |
| What it builds on | `space_app.py`, `reggiebot/` | unchanged (one helper, `source_pages()`, was split out of `with_sources()` so the app can reuse it) |
| The Android app | `android_app/` (Kotlin, Jetpack Compose) | **new** |
| This notebook | `ISU-CampusAssistant-Android.ipynb` | **new** |

The other notebooks and the existing Space (`JakoBob/Campus_Assistant`) are not changed.

### What the app does

* **Chat:** type a question or tap a starter question. The answer appears as it is written, with the pages it
  used listed underneath (tap to open them).
* **Report this answer:** under every answer. Google Play requires apps that generate content with AI to let users
  flag offensive content without leaving the app. Reports go to a private dataset only you can read (§10).
* **Unofficial notice** on the chat screen and the About screen, which also links the privacy policy.
* **No account, no ads, nothing saved on the phone.** The conversation lasts until the app is closed.

### Steps, and who does them

| Step | Section | Who |
|---|---|---|
| Test the app's server on this PC with a tiny model | §1 | run it |
| Build the Space folder | §2 | run it |
| Upload the new Space | §3 | **you**: set `CONTACT_EMAIL`, tick `UPLOAD_SPACE` |
| Check the live server, the way the app calls it | §4 | run it once the Space has started |
| Install the Android build tools on D: (~4 GB) | §5 | **you**: read and accept the Android SDK license |
| Build and test the app, install it on your phone | §6 | run it |
| See it working in an emulator on this PC, before uploading anything | §6b | tick `TRY_IN_EMULATOR` (one-time: approve the emulator's driver) |
| Build the signed release for Google Play | §7 | run it; you choose the upload key's password |
| Store listing text, icon, privacy and safety answers | §8 | run it, then read and edit |
| Publish | §9 | **you**, in the Google Play Console |
| Read reports and app usage | §10 | any time |

### Read before publishing

* **The name is 33 characters, and Google Play titles allow 30.** The app itself uses the full name. The store
  title is `STORE_TITLE` in the settings ("Campus Asst. ISU (unofficial)", 29 characters). Change it if you like.
* **ISU's name.** Google Play rejects apps that look like they come from an organization they don't. "(unofficial)"
  and the notices help, but check with your instructor, or ISU, before publishing publicly.
* **Your contact email** is required by Google Play and by the privacy policy: set `CONTACT_EMAIL`.
* **The privacy policy is a promise you make.** §2 writes it from what the app really does. Read it, and keep it
  true if you change the app.

---
## Settings

In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import time
from datetime import date
from pathlib import Path

# Work from the project folder - the nearest folder holding reggiebot/ - like the other notebooks.
PROJECT = next((folder for folder in (Path.cwd(), *Path.cwd().parents) if (folder / "reggiebot").is_dir()), None)
if PROJECT is None:
    raise FileNotFoundError("No reggiebot/ folder here or above - keep this notebook inside the project folder.")
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
import reggiebot.config                         # also reads HF_TOKEN from .env

# --- the app ---------------------------------------------------------------------------------------------
APP_NAME      = "Campus Assistant ISU (unofficial)"
STORE_TITLE   = "Campus Asst. ISU (unofficial)"    # Google Play titles allow 30 characters; APP_NAME has 33
APP_ID        = "com.jakobob.campusassistant"       # the app's id on Google Play: it can never change once published
VERSION_CODE  = 1                                   # raise by 1 for every upload to Google Play
VERSION_NAME  = "1.0"                               # the version people see
CONTACT_EMAIL = ""                                  # required: shown in the privacy policy and on Google Play

# --- the server ------------------------------------------------------------------------------------------
SPACE_REPO       = "JakoBob/Campus_Assistant_App"   # a NEW Space; JakoBob/Campus_Assistant is not touched
REPORTS_REPO     = "JakoBob/campus-assistant-app-reports"   # a private dataset: reports + the anonymous usage log
SPACE_MODEL_ID   = "google/gemma-4-12B-it"
GPU_SECONDS      = 60
LOCAL_TEST_MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct"   # §1: a tiny model, so the test runs on a CPU

# --- folders: everything big goes on D:, because C: is nearly full -----------------------------------------
TOOLS_DIR   = Path("D:/tools/AndroidTools")     # Java, the Android SDK, the emulator and Gradle's cache (making
                                                # folders at the top of D: needs administrator rights)
BUILD_DIR   = TOOLS_DIR / "builds" / "space"    # what §3 uploads to the Space (~75 MB, rebuilt by §2)
GRADLE_BUILD_DIR = TOOLS_DIR / "builds" / "gradle"   # Gradle's working files for the app (~150 MB)
APP_BUILD   = GRADLE_BUILD_DIR / "app"          # ... where the built app, its test results and bundle appear
ANDROID_DIR = PROJECT / "android_app"           # the Android project
OUTPUT_DIR  = ANDROID_DIR / "outputs"           # the finished app files, copied here to find easily (git ignores them)
KEY_FILE    = TOOLS_DIR / "keys" / "campus-assistant-upload.jks"   # the upload key (§7): back it up

PRIVACY_URL = f"https://huggingface.co/spaces/{SPACE_REPO}/blob/main/PRIVACY.md"


def server_url():
    """The Space's direct address, which the app calls. Asked from Hugging Face once the Space exists."""
    try:
        from huggingface_hub import HfApi
        host = HfApi(token=os.environ.get("HF_TOKEN")).space_info(SPACE_REPO).host
        if host:
            return host.rstrip("/")
    except Exception:
        pass
    return "https://" + re.sub(r"[^a-z0-9]+", "-", SPACE_REPO.lower()).strip("-") + ".hf.space"


assert len(STORE_TITLE) <= 30, f"STORE_TITLE has {len(STORE_TITLE)} characters; Google Play allows 30"
print(f"app:     {APP_NAME}  ({APP_ID}, version {VERSION_NAME}, build {VERSION_CODE})")
print(f"server:  {SPACE_REPO}  ->  {server_url()}")
print(f"reports: {REPORTS_REPO} (private)")
if not CONTACT_EMAIL:
    print("! CONTACT_EMAIL is empty: set it before §2-§3 (the privacy policy and Google Play need one)")

---
## 1 · Test the app's server on this computer

`android_space_app.py` runs here just as it will on the Space, but with a tiny model (SmolLM2-135M, ~270 MB, on the
CPU) instead of Gemma 4. The cell starts it in the background, talks to it **over plain HTTP the way the Android app
does**, prints what comes back, and stops it:

| Endpoint | What the app sends | What comes back |
|---|---|---|
| `status` | nothing | name, model, how fresh the data is, the privacy policy address |
| `ask` | the question, plus the last 4 turns of the conversation | the answer about 3 times a second while it is written; the last update has `done: true`, the status and the source pages |
| `report` | the answer's id, a reason (`offensive`, `wrong`, `other`) and an optional note | `ok: true` once it is saved |

Each call is two requests: `POST /gradio_api/call/<name>` starts it and returns an event id, then
`GET /gradio_api/call/<name>/<id>` streams the result. The server writes its reports and log to a temporary folder
here, never to the project. The tiny model's answers are poor: this checks the wiring, not the quality.

In [ ]:
import requests


def call_endpoint(base, name, data, on_update=None, timeout=600):
    """Call one of the app's endpoints the way the Android app does: POST starts it, then the result streams in."""
    try:
        started = requests.post(f"{base}/gradio_api/call/{name}", json={"data": data}, timeout=60)
        started.raise_for_status()
        event_id = started.json()["event_id"]
    except (requests.RequestException, ValueError, KeyError) as exc:
        raise RuntimeError(f"The server at {base} did not take the request ({type(exc).__name__}). If it is a Space "
                           "that was asleep, it may still be starting: wait a few minutes and run this again.") from exc
    kind, latest = None, None
    with requests.get(f"{base}/gradio_api/call/{name}/{event_id}", stream=True, timeout=timeout) as reply:
        reply.raise_for_status()
        for line in reply.iter_lines(decode_unicode=True):
            if line.startswith("event:"):
                kind = line[6:].strip()
            elif line.startswith("data:") and kind == "error":
                raise RuntimeError(f"{name} failed on the server: {line[5:].strip()[:200]}")
            elif line.startswith("data:") and kind in ("generating", "complete"):
                latest = json.loads(line[5:])[0]
                if kind == "complete":
                    return latest
                if on_update:
                    on_update(latest)
    return latest


class LocalServer:
    """android_space_app.py in the background on this PC, with a throwaway data folder. answer_with: "tiny" (the
    small test model) or "ollama" (Gemma 4 through Ollama, like the Ollama edition: real answers, for §6b)."""

    def __init__(self, port=7862, answer_with="tiny"):
        self.port, self.url, self.answer_with = port, f"http://127.0.0.1:{port}", answer_with
        self.data = Path(tempfile.mkdtemp(prefix="campus-app-server-"))

    def __enter__(self):
        env = dict(os.environ, SPACE_MODEL_ID=LOCAL_TEST_MODEL, GRADIO_SERVER_PORT=str(self.port),
                   APP_DATA_DIR=str(self.data), PRIVACY_URL=PRIVACY_URL, REPORTS_REPO="", HF_TOKEN="",
                   APP_ANSWER_WITH=self.answer_with, PYTHONIOENCODING="utf-8")
        self.log = open(self.data / "server.log", "w", encoding="utf-8")
        self.process = subprocess.Popen([sys.executable, "android_space_app.py"], cwd=PROJECT, env=env,
                                        stdout=self.log, stderr=subprocess.STDOUT)
        for _ in range(300):                       # loading the data and the model takes about a minute
            try:
                if requests.get(self.url, timeout=2).ok:
                    return self
            except requests.RequestException:
                pass
            if self.process.poll() is not None:
                break
            time.sleep(1)
        self.__exit__()
        raise RuntimeError("The server did not start. The end of its log:\n"
                           + (self.data / "server.log").read_text(encoding="utf-8")[-3000:])

    def __exit__(self, *exc):
        self.process.terminate()
        try:
            self.process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            self.process.kill()
        self.log.close()


def show_answer(answer):
    print(f"[{answer['status']}] {answer['text'][:400]}")
    for page in answer["sources"]:
        print(f"   {answer['sources_heading']}: {page['label']}  {page['url']}")


RUN_LOCAL_TEST = True

if RUN_LOCAL_TEST:
    with LocalServer() as server:
        print("status:", call_endpoint(server.url, "status", []))
        updates = []
        answer = call_endpoint(server.url, "ask", ["What are the prerequisites for IT 214?", "[]"], updates.append)
        print(f"\nask: {len(updates)} updates while it was written, then the finished answer:")
        show_answer(answer)
        history = json.dumps([{"role": "user", "content": "What are the prerequisites for IT 214?"},
                              {"role": "assistant", "content": answer["text"]}])
        follow_up = call_endpoint(server.url, "ask", ["How many credit hours is it?", history])
        print("\nfollow-up question:", follow_up["status"])
        print("report:", call_endpoint(server.url, "report", [answer["id"], "wrong", "test from §1", "", ""]))
    written = sorted(p.relative_to(server.data).as_posix() for p in server.data.rglob("*") if p.is_file())
    print("\nthe server wrote, in a temporary folder:", ", ".join(written))
else:
    print("Local test skipped - set RUN_LOCAL_TEST = True to run it.")

---
## 2 · Build the Space folder

Everything the new Space needs, assembled in `BUILD_DIR` (`D:\tools\AndroidTools\builds\space`, because C: is nearly
full):

| In the folder | What it is |
|---|---|
| `android_space_app.py` | the entry point the Space runs |
| `space_app.py`, `reggiebot/` | the Space edition and the package it builds on, copied as they are now |
| `isu_catalog_cache/` | the saved data: `corpus.jsonl`, `snapshot.json` and the Chroma vectors (~67 MB) |
| `requirements.txt` | what the Space installs: the same list as the Space edition |
| `README.md` | the Space's settings (`app_file: android_space_app.py`) and a short description |
| `PRIVACY.md` | **the privacy policy.** Google Play needs its address; it is shown at `PRIVACY_URL` |

Re-run this after changing the code or refreshing the data, then upload again (§3).

In [ ]:
import gradio
from reggiebot.config import CACHE_DIR, CHROMA_DIR

SPACE_REQUIREMENTS = """\
# Installed on the Space. Gradio itself comes from sdk_version in README.md.
spaces                        # ZeroGPU's @spaces.GPU
torch==2.13.0                 # ZeroGPU supports PyTorch 2.8 to 2.13
transformers==5.19.0          # knows Gemma 4 (gemma4_unified)
sentence-transformers==6.1.0
chromadb==1.5.9               # the version that built the saved vector database
rank-bm25>=0.2.2
requests>=2.31
beautifulsoup4>=4.12
lxml>=5.0
trafilatura>=1.6
numpy>=1.26
scikit-learn>=1.3
python-dotenv>=1.0
huggingface_hub               # CommitScheduler: copies reports to the private dataset
"""

SHORT_DESCRIPTION = "Unofficial student assistant for ISU courses and campus"
assert len(SHORT_DESCRIPTION) <= 60, "Hugging Face allows 60 characters"


def space_readme():
    return f"""---
title: {APP_NAME}
emoji: 🎓
colorFrom: red
colorTo: gray
sdk: gradio
sdk_version: {gradio.__version__}
python_version: "3.12"
app_file: android_space_app.py
pinned: false
short_description: {SHORT_DESCRIPTION}
---

# {APP_NAME}

The server for the **{APP_NAME}** Android app, and the same chat on the web. Ask about Illinois State University
courses, billing, housing and dining. Answers come only from ISU's public catalog and websites, with a link to every
source, written by {SPACE_MODEL_ID} on ZeroGPU.

**Unofficial:** an independent student project, not made by, endorsed by or affiliated with Illinois State
University. Answers can be wrong; check anything important with the university.

[Privacy policy](PRIVACY.md)
"""


def privacy_policy():
    contact = CONTACT_EMAIL or "(the developer has not set a contact email yet)"
    return f"""# Privacy policy: {APP_NAME}

Last updated: {date.today():%B %d, %Y}

{APP_NAME} is an independent student project. It is not made by, endorsed by or affiliated with Illinois State
University. This policy covers the Android app and its server, the Hugging Face Space `{SPACE_REPO}`.

## What the app sends

* **Your question**, and the last few questions and answers of the current conversation, so that follow-up
  questions make sense. They are sent over an encrypted connection (HTTPS) to the server, which uses them to write
  the answer and does not store them, unless you report the answer (below).
* **Nothing else.** The app has no account or sign-in, and does not collect your name, email, contacts, location,
  photos, device identifiers or advertising ID. It has no ads and no analytics or tracking libraries. The
  conversation is not saved on your phone.

## What the server keeps

* **An anonymous usage count**, one row per question: the time, a random code for that request, whether it was
  answered, how long it took, the number of retries and the AI model's name. It does not include the question.
* **Reports.** If you tap "Report this answer", the report stores the question, the answer, the reason you chose
  and anything you typed in the note. Please don't put personal information in the note.

Both are kept in a private Hugging Face dataset that only the developer can read. They are used only to fix problems
and improve the answers, are never sold or shared, and are deleted within 12 months.

## Who else handles the data

The server runs on Hugging Face (huggingface.co), which hosts it and may keep standard technical logs, such as IP
addresses, under its own privacy policy: https://huggingface.co/privacy. Answers are written by an AI model
({SPACE_MODEL_ID}) running on that server. No data is sent to any other company.

## Children

The app is meant for university students and is not directed at children under 13.

## Deleting a report, and questions

Reports are not linked to you. To ask about this policy, or to have a report you sent deleted, email {contact} with
the date and the question you asked.

## Changes

If this policy changes, the new version will be posted at this address with a new date.
"""


def build_space():
    """Assemble android_space_build/ from the current code and data. Returns the folder."""
    if BUILD_DIR.exists():
        if any(BUILD_DIR.iterdir()) and not (BUILD_DIR / "android_space_app.py").exists():
            raise RuntimeError(f"{BUILD_DIR} exists and is not a Space build folder - not deleting it.")
        shutil.rmtree(BUILD_DIR)
    BUILD_DIR.mkdir(parents=True)
    for name in ("android_space_app.py", "space_app.py"):
        shutil.copy2(PROJECT / name, BUILD_DIR / name)
    shutil.copytree(PROJECT / "reggiebot", BUILD_DIR / "reggiebot", ignore=shutil.ignore_patterns("__pycache__", "*.pyc"))
    data = BUILD_DIR / "isu_catalog_cache"
    data.mkdir()
    for name in ("corpus.jsonl", "snapshot.json"):
        shutil.copy2(CACHE_DIR / name, data / name)
    shutil.copytree(CHROMA_DIR, data / "chroma")
    (BUILD_DIR / "requirements.txt").write_text(SPACE_REQUIREMENTS, encoding="utf-8")
    (BUILD_DIR / "README.md").write_text(space_readme(), encoding="utf-8")
    (BUILD_DIR / "PRIVACY.md").write_text(privacy_policy(), encoding="utf-8")
    return BUILD_DIR


build_space()
files = [p for p in BUILD_DIR.rglob("*") if p.is_file()]
for path in sorted(files):
    if path.stat().st_size > 1_000_000 or path.parent == BUILD_DIR:
        print(f"{path.relative_to(BUILD_DIR).as_posix():<55} {path.stat().st_size / 1e6:8.1f} MB")
print(f"{'total':<55} {sum(p.stat().st_size for p in files) / 1e6:8.1f} MB in {len(files)} files")
if not CONTACT_EMAIL:
    print("\n! PRIVACY.md has no contact email yet: set CONTACT_EMAIL in the settings and run this cell again")

---
## 3 · Upload the new Space (you run this)

1. **Account and token:** the same as for the Space edition: a write token in `.env` as `HF_TOKEN=...`.
   *Safer:* create a **fine-grained** token that can only write to `REPORTS_REPO`, and put it in `.env` as
   `HF_REPORTS_TOKEN=...`. The Space keeps that token as a secret, to save reports; without it, it gets `HF_TOKEN`.
2. **Two ZeroGPU Spaces:** a free account can host two. This is your second (the first is `JakoBob/Campus_Assistant`).
3. **Public:** the app calls the Space without signing in, so the Space must be public.
4. **Tick `UPLOAD_SPACE`** and run the cell. It creates the private reports dataset and the Space, sets the
   variables and the secret, asks for ZeroGPU, and uploads `BUILD_DIR`.
5. **Wait** for the first start (several minutes: it installs the packages and downloads Gemma 4). **Logs** on the
   Space's page ends with `loaded: google/gemma-4-12B-it ...` and
   `[reports] copied to the private dataset ... every 5 minutes`.

In [ ]:
UPLOAD_SPACE = False

if not UPLOAD_SPACE:
    print("Upload skipped - set CONTACT_EMAIL, tick UPLOAD_SPACE and run this cell.")
elif not CONTACT_EMAIL:
    print("Set CONTACT_EMAIL in the settings first, then rebuild (§2): the privacy policy needs it.")
elif not os.environ.get("HF_TOKEN"):
    print("No HF_TOKEN found. Add HF_TOKEN=<a write token from huggingface.co/settings/tokens> to .env and "
          "restart the kernel.")
elif not (BUILD_DIR / "android_space_app.py").exists() or CONTACT_EMAIL not in (BUILD_DIR / "PRIVACY.md").read_text(encoding="utf-8"):
    print("Build the Space folder first (§2), after setting CONTACT_EMAIL.")
else:
    from huggingface_hub import HfApi
    api = HfApi(token=os.environ["HF_TOKEN"])
    api.create_repo(REPORTS_REPO, repo_type="dataset", private=True, exist_ok=True)
    api.create_repo(SPACE_REPO, repo_type="space", space_sdk="gradio", private=False, exist_ok=True)
    for key, value in {"SPACE_MODEL_ID": SPACE_MODEL_ID, "SPACE_GPU_SECONDS": str(GPU_SECONDS),
                       "REPORTS_REPO": REPORTS_REPO, "PRIVACY_URL": PRIVACY_URL}.items():
        api.add_space_variable(SPACE_REPO, key, value)
    api.add_space_secret(SPACE_REPO, "HF_TOKEN", os.environ.get("HF_REPORTS_TOKEN") or os.environ["HF_TOKEN"])
    try:
        api.request_space_hardware(SPACE_REPO, "zero-a10g")      # Hugging Face's id for ZeroGPU hardware
        print("ZeroGPU requested.")
    except Exception as exc:
        print(f"Could not request ZeroGPU from here ({type(exc).__name__}: {str(exc)[:120]}).")
        print(f"Pick it by hand: https://huggingface.co/spaces/{SPACE_REPO}/settings -> Hardware -> ZeroGPU")
    api.upload_folder(folder_path=str(BUILD_DIR), repo_id=SPACE_REPO, repo_type="space",
                      commit_message=f"Update {APP_NAME}")
    print(f"Uploaded. Watch it start: https://huggingface.co/spaces/{SPACE_REPO} (open Logs)")
    print(f"The app will call:  {server_url()}")
    print(f"Privacy policy:     {PRIVACY_URL}")

---
## 4 · Check the live server, the way the app calls it

Once the Space shows **Running**: `status`, then one question to Gemma 4. It uses a few seconds of your daily
ZeroGPU time. If the Space was asleep, the first call wakes it, which can take a few minutes; run it again then.

In [ ]:
CHECK_LIVE = False

if CHECK_LIVE:
    url = server_url()
    print("server:", url)
    print("status:", call_endpoint(url, "status", [], timeout=900))
    started = time.time()
    show_answer(call_endpoint(url, "ask", ["When will I get my refund?", "[]"], timeout=900))
    print(f"\n({time.time() - started:.0f} s)")
else:
    print("Skipped - tick CHECK_LIVE once the Space is running.")

---
## 5 · Install the Android build tools on D: (you accept the license)

Building an Android app needs **Java 17** and the **Android SDK** (Google's build tools). Android Studio is not
needed: everything runs from this notebook. All of it goes in `TOOLS_DIR` on D: (~4 GB), because C: is nearly full.

| Tool | From | Size |
|---|---|---|
| Java 17 (Eclipse Temurin) | adoptium.net | ~180 MB download |
| Android SDK command-line tools | dl.google.com | ~150 MB |
| Android 16 platform, build tools, platform-tools (`adb`) | installed by the SDK tools | ~300 MB |
| Gradle 8.14.3 and the app's libraries | downloaded by the first build (§6) | ~1–2 GB |

**The Android SDK has a license** that you must accept before it installs:
[developer.android.com/studio/terms](https://developer.android.com/studio/terms). Read it, then tick
`ACCEPT_ANDROID_SDK_LICENSE` together with `INSTALL_TOOLS`. Running it again skips what is already installed.

In [ ]:
INSTALL_TOOLS = False
ACCEPT_ANDROID_SDK_LICENSE = False

JDK_URL = ("https://github.com/adoptium/temurin17-binaries/releases/download/jdk-17.0.21%2B9/"
           "OpenJDK17U-jdk_x64_windows_hotspot_17.0.21_9.zip")
SDK_TOOLS_URL = "https://dl.google.com/android/repository/commandlinetools-win-13114758_latest.zip"
SDK_PACKAGES = ["platform-tools", "platforms;android-36", "build-tools;35.0.0"]

JAVA_HOME = TOOLS_DIR / "jdk-17"
ANDROID_HOME = TOOLS_DIR / "sdk"
SDKMANAGER = ANDROID_HOME / "cmdline-tools" / "latest" / "bin" / "sdkmanager.bat"
ADB = ANDROID_HOME / "platform-tools" / "adb.exe"


def tool_env(**extra):
    """Environment for the Android tools: Java, the SDK, Gradle's cache and the app's build output, all on D:."""
    env = dict(os.environ, JAVA_HOME=str(JAVA_HOME), ANDROID_HOME=str(ANDROID_HOME), ANDROID_SDK_ROOT=str(ANDROID_HOME),
               GRADLE_USER_HOME=str(TOOLS_DIR / "gradle-home"), ANDROID_USER_HOME=str(TOOLS_DIR / "android-user"),
               CA_BUILD_DIR=str(GRADLE_BUILD_DIR))
    env["PATH"] = os.pathsep.join([str(JAVA_HOME / "bin"), str(ANDROID_HOME / "platform-tools"), env["PATH"]])
    env.update(extra)
    return env


def tools_ready():
    return (JAVA_HOME / "bin" / "java.exe").exists() and SDKMANAGER.exists() and (ANDROID_HOME / "platforms" / "android-36").exists()


def write_app_properties(url):
    """The settings the app is built with (§6, §7): its server, privacy policy, id and version."""
    (ANDROID_DIR / "app.properties").write_text(
        "# Written by ISU-CampusAssistant-Android.ipynb from its settings - change them there, not here.\n"
        f"serverUrl={url}\nprivacyUrl={PRIVACY_URL}\napplicationId={APP_ID}\n"
        f"versionCode={VERSION_CODE}\nversionName={VERSION_NAME}\n", encoding="utf-8")


def gradle(*tasks, **extra_env):
    """Run Gradle in android_app/ with the tools on D:, and show the end of its output."""
    result = subprocess.run([str(ANDROID_DIR / "gradlew.bat"), *tasks, "--console=plain"], cwd=ANDROID_DIR,
                            env=tool_env(**extra_env), capture_output=True, text=True, encoding="utf-8",
                            errors="replace", timeout=3600)
    output = (result.stdout + result.stderr).strip().splitlines()
    print("\n".join(line for line in output[-(80 if result.returncode else 12):] if line.strip()))
    if result.returncode:
        raise RuntimeError(f"gradle {' '.join(tasks)} failed - the reason is in the lines above")
    return output


def download(url, target):
    """Download to a file, showing progress every 10%."""
    with requests.get(url, stream=True, timeout=60) as reply:
        reply.raise_for_status()
        total, done, shown = int(reply.headers.get("content-length", 0)), 0, -10
        with open(target, "wb") as f:
            for chunk in reply.iter_content(1 << 20):
                f.write(chunk)
                done += len(chunk)
                if total and done * 100 // total >= shown + 10:
                    shown = done * 100 // total
                    print(f"   {target.name}: {shown}% of {total / 1e6:.0f} MB", flush=True)


def unpack(url, folder):
    """Download a zip that holds one top folder, and unpack that folder's contents into `folder`."""
    import zipfile
    work = Path(tempfile.mkdtemp(prefix="android-tools-", dir=TOOLS_DIR))
    try:
        archive = work / "download.zip"
        download(url, archive)
        with zipfile.ZipFile(archive) as z:
            z.extractall(work / "unpacked")
        (top,) = (work / "unpacked").iterdir()
        folder.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(top), str(folder))
    finally:
        shutil.rmtree(work, ignore_errors=True)


def install_tools():
    TOOLS_DIR.mkdir(parents=True, exist_ok=True)
    if not (JAVA_HOME / "bin" / "java.exe").exists():
        print("Java 17 ...")
        unpack(JDK_URL, JAVA_HOME)
    if not SDKMANAGER.exists():
        print("Android SDK command-line tools ...")
        unpack(SDK_TOOLS_URL, SDKMANAGER.parents[1])
    print("accepting the Android SDK license ...")
    subprocess.run([str(SDKMANAGER), f"--sdk_root={ANDROID_HOME}", "--licenses"], input="y\n" * 30, text=True,
                   env=tool_env(), capture_output=True, timeout=600)
    packages = TOOLS_DIR / "sdk-packages.txt"            # a file, because ';' in names upsets Windows' cmd
    packages.write_text("\n".join(SDK_PACKAGES) + "\n", encoding="utf-8")
    print("installing", ", ".join(SDK_PACKAGES), "...")
    result = subprocess.run([str(SDKMANAGER), f"--sdk_root={ANDROID_HOME}", f"--package_file={packages}"],
                            env=tool_env(), capture_output=True, text=True, timeout=1800)
    if result.returncode:
        raise RuntimeError("sdkmanager failed:\n" + (result.stdout + result.stderr)[-2000:])
    java = subprocess.run([str(JAVA_HOME / "bin" / "java.exe"), "-version"], capture_output=True, text=True)
    print("ready:", java.stderr.splitlines()[0], "| SDK in", ANDROID_HOME)


if tools_ready():
    print(f"The Android build tools are installed in {TOOLS_DIR}.")
elif not INSTALL_TOOLS:
    print("Not installed yet - read the license above, then tick INSTALL_TOOLS and ACCEPT_ANDROID_SDK_LICENSE.")
elif not ACCEPT_ANDROID_SDK_LICENSE:
    print("Read https://developer.android.com/studio/terms and tick ACCEPT_ANDROID_SDK_LICENSE to install.")
else:
    install_tools()

---
## 6 · Build and test the app

1. **Settings into the app:** writes `android_app/app.properties` with the server's address, the privacy policy's
   address, the app id and the version. The app reads them when it is built.
2. **Tests on this computer:** the answer-formatting reader and the server's message format (`MarkdownTest`,
   `CampusServerTest`). With `TEST_WITH_LOCAL_SERVER`, the app's own networking code also talks to the §1 server:
   status, a streamed answer and a report, exactly as on a phone.
3. **The test app:** `android_app/outputs/campus-assistant-<version>-test.apk`.

The first build downloads Gradle and the app's libraries (a few minutes); later builds take under a minute.

**On your phone:** turn on **Developer options → USB debugging**, plug it in and allow this PC; the cell installs
the app. Or copy the `.apk` to the phone and open it (Android asks you to allow installing apps from that source).
The test app talks to the real Space, so do this after §3.

In [ ]:
TEST_WITH_LOCAL_SERVER = True
INSTALL_ON_PHONE = True


def unit_test_counts():
    import xml.etree.ElementTree as ET
    totals = {"tests": 0, "failures": 0, "errors": 0, "skipped": 0}
    for report in (APP_BUILD / "test-results" / "testDebugUnitTest").glob("*.xml"):
        root = ET.parse(report).getroot()
        for key in totals:
            totals[key] += int(root.get(key, 0))
    return totals


if not tools_ready():
    print("Install the Android build tools first (§5).")
else:
    url = server_url()
    write_app_properties(url)
    print(f"app.properties: server {url}\n")
    if TEST_WITH_LOCAL_SERVER:
        with LocalServer() as server:
            gradle("testDebugUnitTest", f"-PcaTestServer={server.url}")
    else:
        gradle("testDebugUnitTest")
    print("\nunit tests:", unit_test_counts(), "\n")

    gradle("assembleDebug")
    OUTPUT_DIR.mkdir(exist_ok=True)
    apk = OUTPUT_DIR / f"campus-assistant-{VERSION_NAME}-test.apk"
    shutil.copy2(APP_BUILD / "outputs" / "apk" / "debug" / "app-debug.apk", apk)
    print(f"\ntest app: {apk} ({apk.stat().st_size / 1e6:.1f} MB)")

    phones = [line.split()[0] for line in subprocess.run([str(ADB), "devices"], capture_output=True, text=True)
              .stdout.splitlines()[1:] if line.strip().endswith("device") and not line.startswith("emulator-")]
    if INSTALL_ON_PHONE and phones:                 # a real phone on USB, never the emulator (§6b has its own copy)
        print(subprocess.run([str(ADB), "-s", phones[0], "install", "-r", str(apk)], capture_output=True,
                             text=True).stdout.strip())
        print("Installed. Open 'Campus Assistant' on the phone.")
    else:
        print("No phone connected with USB debugging - copy the .apk to the phone and open it to install.")

---
## 6b · Try it in an emulator on this PC

See the app working before anything is uploaded: a virtual **Pixel 7 running Android 16**, in a window on this PC,
with the Play Store and Chrome like a real phone. The app talks to the server running here, which answers with
**Gemma 4 through Ollama** (the Ollama edition's model), so the answers are real. **Start Ollama first.**

| What | Where (all on D:) |
|---|---|
| The emulator and the Android 16 image (~3 GB) | `TOOLS_DIR/sdk` |
| The virtual phone, `CampusAssistant_Pixel7` | `TOOLS_DIR/android-user/avd` |
| The emulator copy of the app | `android_app/outputs/campus-assistant-<version>-emulator.apk`. It talks to this PC, so never upload it |
| Screenshots (usable for the store listing) | `android_app/store/screenshots/` |

**One time only: let the emulator use the CPU's virtualization.** Without it the emulator won't start. If the cell
says so, double-click
`D:\tools\AndroidTools\sdk\extras\google\Android_Emulator_Hypervisor_Driver\silent_install.bat` and choose **Yes**
when Windows asks. No restart is needed. (To remove it later: run the same file with `-u`.)

Tick `TRY_IN_EMULATOR` and run the first cell. It starts the server, builds the app, opens the emulator window and
opens the app (the first start of the virtual phone takes a minute or two). Then try it: tap a starter question,
or click the text box and type with the PC keyboard. The next cell saves a screenshot; the last one stops everything.

**Without the notebook:** once this cell has run once, double-click **`start_emulator.bat`** in the project folder to
open everything (Ollama must be running), and **`stop_emulator.bat`** to close it.

In [ ]:
TRY_IN_EMULATOR = False
ANSWER_WITH = "ollama"        # "ollama": Gemma 4 through Ollama (start it first); "tiny": the §1 test model, poor answers

EMULATOR_AVD   = "CampusAssistant_Pixel7"
EMULATOR_IMAGE = "system-images;android-36;google_apis_playstore;x86_64"
EMULATOR_PORT  = 7863                       # the server for the emulator (§1 and §6 use 7862)
EMULATOR       = ANDROID_HOME / "emulator" / "emulator.exe"
AVD_HOME       = TOOLS_DIR / "android-user" / "avd"
DRIVER_INSTALLER = ANDROID_HOME / "extras" / "google" / "Android_Emulator_Hypervisor_Driver" / "silent_install.bat"
SCREENSHOT_DIR = ANDROID_DIR / "store" / "screenshots"


def set_up_emulator():
    """Install the emulator, the Android 16 image and the driver's installer, and create the virtual phone, if missing.
    Returns (can run, what the emulator says about acceleration)."""
    image = ANDROID_HOME / "system-images" / "android-36" / "google_apis_playstore" / "x86_64" / "system.img"
    missing = [name for name, there in [("emulator", EMULATOR.exists()), (EMULATOR_IMAGE, image.exists()),
                                        ("extras;google;Android_Emulator_Hypervisor_Driver", DRIVER_INSTALLER.exists())]
               if not there]
    if missing:
        print("installing", ", ".join(missing), "(about 2 GB) ...")
        packages = TOOLS_DIR / "emulator-packages.txt"
        packages.write_text("\n".join(missing) + "\n", encoding="utf-8")
        subprocess.run([str(SDKMANAGER), f"--sdk_root={ANDROID_HOME}", f"--package_file={packages}"], env=tool_env(),
                       capture_output=True, text=True, timeout=3600, check=True)
    if not (AVD_HOME / f"{EMULATOR_AVD}.avd").exists():
        avdmanager = SDKMANAGER.with_name("avdmanager.bat")
        subprocess.run(f'"{avdmanager}" create avd --force -n {EMULATOR_AVD} -k "{EMULATOR_IMAGE}" -d pixel_7',
                       input="no\n", text=True, capture_output=True, env=tool_env(ANDROID_AVD_HOME=str(AVD_HOME)),
                       check=True)
        config = AVD_HOME / f"{EMULATOR_AVD}.avd" / "config.ini"
        lines = config.read_text(encoding="utf-8").splitlines()
        lines = ["hw.keyboard=yes" if l.startswith("hw.keyboard=") else "hw.ramSize=4G" if l.startswith("hw.ramSize=")
                 else l for l in lines]                 # type with the PC keyboard; 4 GB so Android 16 runs smoothly
        config.write_text("\n".join(lines) + "\n", encoding="utf-8")
        print(f"created the virtual phone {EMULATOR_AVD} in {AVD_HOME}")
    check = subprocess.run([str(EMULATOR), "-accel-check"], env=tool_env(), capture_output=True, text=True)
    return check.returncode == 0, (check.stdout + check.stderr).strip()


def adb(*args, timeout=120):
    return subprocess.run([str(ADB), "-e", *args], env=tool_env(), capture_output=True, text=True, timeout=timeout)


def emulator_running():
    devices = subprocess.run([str(ADB), "devices"], env=tool_env(), capture_output=True, text=True).stdout
    return any(line.startswith("emulator-") and line.strip().endswith("device") for line in devices.splitlines())


def start_emulator():
    """Open the emulator window (if it isn't open) and wait until Android has finished starting."""
    if not emulator_running():
        log = open(TOOLS_DIR / "emulator.log", "w", encoding="utf-8")
        subprocess.Popen([str(EMULATOR), "-avd", EMULATOR_AVD, "-netdelay", "none", "-netspeed", "full"],
                         env=tool_env(ANDROID_AVD_HOME=str(AVD_HOME)), stdout=log, stderr=subprocess.STDOUT)
        print("emulator window opening - the first start of the virtual phone takes a minute or two ...")
    adb("wait-for-device", timeout=600)
    for _ in range(300):
        if adb("shell", "getprop", "sys.boot_completed").stdout.strip() == "1":
            return
        time.sleep(2)
    raise RuntimeError(f"Android did not finish starting in the emulator. Its log: {TOOLS_DIR / 'emulator.log'}")


if not TRY_IN_EMULATOR:
    print("Skipped - tick TRY_IN_EMULATOR to see the app in an emulator on this PC.")
elif not tools_ready():
    print("Install the Android build tools first (§5).")
else:
    can_run, accel = set_up_emulator()
    if not can_run:
        print("The emulator can't use the CPU's virtualization yet. What it says:\n   " + accel.replace("\n", "\n   "))
        print(f"\nOne-time fix, no restart: double-click\n   {DRIVER_INSTALLER}\n"
              "and choose Yes when Windows asks. Then run this cell again.")
    else:
        # 1. the server, left running while you try the app
        if "EMULATOR_SERVER" not in globals() or EMULATOR_SERVER.process.poll() is not None:
            EMULATOR_SERVER = LocalServer(port=EMULATOR_PORT, answer_with=ANSWER_WITH).__enter__()
        print("server:", call_endpoint(EMULATOR_SERVER.url, "status", []))
        # 2. the app, pointed at this PC (10.0.2.2 is this PC, seen from inside the emulator)
        write_app_properties(server_url())
        gradle("assembleDebug", f"-PcaServerUrl=http://10.0.2.2:{EMULATOR_PORT}")
        OUTPUT_DIR.mkdir(exist_ok=True)
        emulator_apk = OUTPUT_DIR / f"campus-assistant-{VERSION_NAME}-emulator.apk"
        shutil.copy2(APP_BUILD / "outputs" / "apk" / "debug" / "app-debug.apk", emulator_apk)
        # 3. the emulator, with the app installed and open
        start_emulator()
        print(adb("install", "-r", str(emulator_apk)).stdout.strip())
        adb("shell", "am", "start", "-n", f"{APP_ID}/com.jakobob.campusassistant.MainActivity")
        print(f"\nThe app is open in the emulator window. Answers come from {ANSWER_WITH} on this PC.")
        print("Leave this running while you try it; the last cell of §6b stops the emulator and the server.")

In [ ]:
# A screenshot of the emulator's screen, saved for the store listing and shown here. Run it whenever you like.
import io
from IPython.display import Image, display
from PIL import Image as Picture

if TRY_IN_EMULATOR and tools_ready() and emulator_running():
    SCREENSHOT_DIR.mkdir(parents=True, exist_ok=True)
    shot = SCREENSHOT_DIR / f"screenshot-{time.strftime('%Y%m%d-%H%M%S')}.png"
    screen = Picture.open(io.BytesIO(subprocess.run([str(ADB), "-e", "exec-out", "screencap", "-p"], env=tool_env(),
                                                    capture_output=True, timeout=60).stdout))
    extra = screen.height - 2 * screen.width        # Google Play allows at most 2:1; the Pixel 7 screen is 20:9
    if extra > 0:                                   # trim the status bar and the gesture bar evenly
        screen = screen.crop((0, extra // 2, screen.width, screen.height - (extra - extra // 2)))
    screen.save(shot)
    print(f"saved {shot} ({screen.width} x {screen.height})")
    display(Image(filename=str(shot), width=320))
else:
    print("The emulator is not running - start it with the cell above.")

In [ ]:
STOP_EMULATOR = False      # tick and run when you are done trying the app

if STOP_EMULATOR:
    if tools_ready() and emulator_running():
        adb("emu", "kill")
        print("emulator closed")
    if "EMULATOR_SERVER" in globals() and EMULATOR_SERVER.process.poll() is None:
        EMULATOR_SERVER.__exit__()
        print("server stopped")
else:
    print("Still running (if it was started). Tick STOP_EMULATOR and run this cell to close the emulator and the server.")

---
## 7 · Build the release for Google Play

Google Play takes an **Android App Bundle** (`.aab`) signed with your **upload key**. The cell creates the key the
first time (in `KEY_FILE` on D:, outside the project), asks for its password, and builds
`android_app/outputs/campus-assistant-<version>-<build>.aab`.

* **Back up the key file and its password** (for example on a USB stick and in a password manager). Google Play
  keeps the key that signs the app for users (Play App Signing, the default); the upload key only proves an update
  comes from you. If it is lost, Google can reset it, but it takes days.
* The password is never written into the project. To avoid typing it, put `ANDROID_UPLOAD_KEY_PASSWORD=...` in `.env`.
* **Every new upload needs a higher `VERSION_CODE`** in the settings.

In [ ]:
BUILD_RELEASE = False

if not BUILD_RELEASE:
    print("Skipped - tick BUILD_RELEASE to build the .aab for Google Play.")
elif not tools_ready():
    print("Install the Android build tools first (§5).")
elif not CONTACT_EMAIL:
    print("Set CONTACT_EMAIL first: Google Play needs it, and so does the privacy policy the app links to.")
else:
    import getpass
    password = os.environ.get("ANDROID_UPLOAD_KEY_PASSWORD") or getpass.getpass("Upload key password (6+ characters): ")
    if len(password) < 6:
        raise ValueError("The password needs at least 6 characters.")
    keys = dict(CA_UPLOAD_STORE_FILE=str(KEY_FILE), CA_UPLOAD_STORE_PASSWORD=password, CA_UPLOAD_KEY_ALIAS="upload")
    if not KEY_FILE.exists():
        KEY_FILE.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run([str(JAVA_HOME / "bin" / "keytool.exe"), "-genkeypair", "-keystore", str(KEY_FILE),
                        "-alias", "upload", "-keyalg", "RSA", "-keysize", "2048", "-validity", "10000",
                        "-storepass:env", "CA_UPLOAD_STORE_PASSWORD", "-keypass:env", "CA_UPLOAD_STORE_PASSWORD",
                        "-dname", "CN=Campus Assistant ISU unofficial"], env=tool_env(**keys), check=True,
                       capture_output=True)
        print(f"Created the upload key: {KEY_FILE}  <- back this file up, with its password")
    write_app_properties(server_url())
    gradle("bundleRelease", **keys)
    OUTPUT_DIR.mkdir(exist_ok=True)
    bundle = OUTPUT_DIR / f"campus-assistant-{VERSION_NAME}-{VERSION_CODE}.aab"
    shutil.copy2(APP_BUILD / "outputs" / "bundle" / "release" / "app-release.aab", bundle)
    print(f"\nFor Google Play: {bundle} ({bundle.stat().st_size / 1e6:.1f} MB)")

---
## 8 · Store listing materials

Writes `android_app/store/` with what the Play Console asks for: the listing text (with Google's length limits
checked), draft answers for the **Data safety** and **content rating** forms, the 512 × 512 icon and the
1024 × 500 feature graphic. **Read and edit them before pasting**: the answers describe what the app does today.

Screenshots come from your phone: install the test app (§6), ask a couple of questions, and take at least two.

In [ ]:
from PIL import Image, ImageDraw, ImageFont

STORE_DIR = ANDROID_DIR / "store"
SHORT_STORE_DESCRIPTION = "Unofficial student app: ISU courses, billing, housing and dining answers"
FULL_STORE_DESCRIPTION = f"""{APP_NAME} answers questions about Illinois State University courses, billing, housing and dining, in plain language, with a link to every source.

UNOFFICIAL: this is an independent student project. It is not made by, endorsed by or affiliated with Illinois State University.

What you can ask
• What a course covers and its prerequisites ("What are the prerequisites for IT 214?")
• Which courses teach a topic ("Which courses teach machine learning?")
• Paying your bill, payment plans and refunds
• Housing options and meal plans

How it works
Every answer is written by an AI model from pages saved from ISU's public course catalog and websites, and lists the pages it used, so you can check them. It cannot see anyone's student account, so it never knows your balance, schedule or housing assignment.

Good to know
• Answers can be wrong. Check anything important, like deadlines and costs, with the university.
• If an answer is offensive, harmful or wrong, tap "Report this answer" under it.
• No account or sign-in, and no ads. The conversation is not saved on your phone.
• After a quiet period the first answer can take a few minutes while the server starts; after that, answers take seconds. Each person has a limited number of answers per day.
"""

DATA_SAFETY = f"""# Data safety form: draft answers

Check each against the current Play Console form. These describe the app as built today.

| Question | Answer |
|---|---|
| Does your app collect or share any of the required user data types? | **Yes** |
| Is all of the user data collected by your app encrypted in transit? | **Yes** (HTTPS only) |
| Do you provide a way for users to request that their data is deleted? | **Yes**: by email to {CONTACT_EMAIL or "CONTACT_EMAIL"} (see the privacy policy) |

## Data types

| Data type | Collected | Shared | Processed ephemerally | Required or optional | Purposes |
|---|---|---|---|---|---|
| App activity → **Other user-generated content** (the questions; and in reports, the question, answer and note) | Yes | No | No: reports are stored | Optional: answering needs the question, but storing happens only when the user reports | App functionality; Fraud prevention, security and compliance |
| App activity → **App interactions** (anonymous count: time, status, speed of each question) | Yes | No | No | Required | Analytics |

Not collected: name, email, user IDs, location, contacts, photos, files, device or other IDs, financial info,
health, messages, web browsing, audio, calendar. Hugging Face hosts the server as a service provider, which Google
does not count as sharing.

## Other Play Console forms

| Form | Answer |
|---|---|
| Privacy policy | {PRIVACY_URL} |
| App access | All functionality is available without special access (no login) |
| Ads | No, the app does not contain ads |
| Target audience | 18 and over (simplest), or 16–17 and 18+. Do **not** pick any age under 13 |
| News app | No |
| Government app | No (it is not made by or for a government or university) |
| Financial features | None |
| Health | None |
| Content rating | see content-rating.md |
"""

CONTENT_RATING = """# Content rating questionnaire (IARC): notes

* **Category:** Reference, News, or Educational.
* Violence, fear, sexuality, language, controlled substances, crude humor, gambling: **No**.
* Users interact or exchange content with each other: **No** (each person talks only to the assistant).
* Shares the user's location: **No**. Digital purchases: **No**.
* If asked about AI-generated content: **Yes**. The app shows text answers written by an AI model, limited to ISU's
  public catalog and websites, with in-app reporting ("Report this answer").
"""


def font(size, bold=False):
    for name in (("segoeuib.ttf", "arialbd.ttf") if bold else ("segoeui.ttf", "arial.ttf")):
        try:
            return ImageFont.truetype(name, size)
        except OSError:
            pass
    return ImageFont.load_default(size)


def draw_bubble(image, left, top, size):
    """The app icon's white speech bubble with three red dots (the same shape as the launcher icon)."""
    s = size / 108
    d = ImageDraw.Draw(image)
    p = lambda x, y: (left + x * s, top + y * s)
    d.rounded_rectangle([p(30, 32), p(78, 70)], radius=8 * s, fill="white")
    d.polygon([p(50, 69), p(39, 79), p(41, 69)], fill="white")
    for x in (43, 54, 65):
        d.ellipse([p(x - 4, 47), p(x + 4, 55)], fill="#CC0000")


def store_materials():
    STORE_DIR.mkdir(parents=True, exist_ok=True)
    checks = {"title": (STORE_TITLE, 30), "short description": (SHORT_STORE_DESCRIPTION, 80),
              "full description": (FULL_STORE_DESCRIPTION, 4000)}
    for label, (text, limit) in checks.items():
        assert len(text) <= limit, f"the {label} has {len(text)} characters; Google Play allows {limit}"
    (STORE_DIR / "listing.md").write_text(
        f"# Store listing\n\n## App name ({len(STORE_TITLE)}/30)\n\n{STORE_TITLE}\n\n"
        f"## Short description ({len(SHORT_STORE_DESCRIPTION)}/80)\n\n{SHORT_STORE_DESCRIPTION}\n\n"
        f"## Full description ({len(FULL_STORE_DESCRIPTION)}/4000)\n\n{FULL_STORE_DESCRIPTION}\n"
        f"## Contact email\n\n{CONTACT_EMAIL or '(set CONTACT_EMAIL in the notebook)'}\n\n"
        f"## Category\n\nEducation\n\n## Privacy policy\n\n{PRIVACY_URL}\n", encoding="utf-8")
    (STORE_DIR / "data-safety.md").write_text(DATA_SAFETY, encoding="utf-8")
    (STORE_DIR / "content-rating.md").write_text(CONTENT_RATING, encoding="utf-8")

    icon = Image.new("RGB", (512, 512), "#CC0000")
    draw_bubble(icon, 0, 0, 512)
    icon.save(STORE_DIR / "icon-512.png")

    graphic = Image.new("RGB", (1024, 500), "#CC0000")
    draw_bubble(graphic, 40, 70, 360)
    d = ImageDraw.Draw(graphic)
    d.text((400, 150), "Campus Assistant ISU", font=font(56, bold=True), fill="white")
    d.text((400, 225), "(unofficial)", font=font(40), fill="white")
    d.text((400, 300), "Courses, billing, housing and dining,", font=font(30), fill="white")
    d.text((400, 340), "with a link to every source", font=font(30), fill="white")
    graphic.save(STORE_DIR / "feature-graphic-1024x500.png")
    return sorted(p.name for p in STORE_DIR.iterdir())


print("written to android_app/store/:", ", ".join(store_materials()))
print(f"title {len(STORE_TITLE)}/30 · short description {len(SHORT_STORE_DESCRIPTION)}/80 · "
      f"full description {len(FULL_STORE_DESCRIPTION)}/4000")

---
## 9 · Publish on Google Play (you do this, in the Play Console)

1. **Developer account:** [play.google.com/console](https://play.google.com/console), a one-time $25 fee and an
   identity check. A personal account is fine.
2. **Create the app:** name `STORE_TITLE`, default language English, **App**, **Free**. Accept the declarations.
3. **Set up the store listing** from `android_app/store/listing.md`, the icon, the feature graphic and at least two
   phone screenshots.
4. **App content:** the privacy policy address, app access, ads, content rating, target audience, data safety and
   the other forms, from `android_app/store/data-safety.md` and `content-rating.md`.
5. **Internal testing** (fastest, up to 100 people): **Testing → Internal testing → Create release**, upload the
   `.aab` from §7, and add testers by email. They install it from a Play Store link, usually within an hour.
   This is enough for a class.
6. **Closed testing** before going public: new personal accounts need **at least 12 testers opted in for 14 days in
   a row** before Google allows a production release. Then **apply for production access**, and Google reviews the app.
7. **Updates:** raise `VERSION_CODE` (and `VERSION_NAME`), re-run §7, and upload the new `.aab`.

| Keep safe | Where |
|---|---|
| The upload key and its password | `KEY_FILE` on D: |
| The app id | `APP_ID`: it can never change once published |
| The Space and the reports dataset | `SPACE_REPO`, `REPORTS_REPO` |

---
## 10 · Read reports and app usage

The Space copies `reports/` and `usage_log.csv` to the private dataset `REPORTS_REPO` every 5 minutes. This
downloads them to a temporary folder and shows the latest. Google expects reports of offensive content to be looked
at and acted on, so check this regularly once the app is out.

In [ ]:
READ_REPORTS = False

if not READ_REPORTS:
    print("Skipped - tick READ_REPORTS to read the reports.")
else:
    import csv
    from collections import Counter
    from huggingface_hub import snapshot_download
    folder = Path(snapshot_download(REPORTS_REPO, repo_type="dataset", token=os.environ.get("HF_TOKEN"),
                                    local_dir=tempfile.mkdtemp(prefix="campus-reports-")))
    reports = [json.loads(line) for path in sorted(folder.glob("reports/*.jsonl"))
               for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    print(f"{len(reports)} reports:", dict(Counter(r["reason"] for r in reports)))
    for r in reports[-20:]:
        print(f"\n{r['time'][:16]}  {r['reason']}  ({r['status'] or 'status unknown'})")
        print(f"   Q: {r['question'][:150]}")
        print(f"   A: {r['answer'][:300]}")
        if r["note"]:
            print(f"   note: {r['note']}")
    usage = folder / "usage_log.csv"
    if usage.exists():
        with open(usage, encoding="utf-8", newline="") as f:
            rows = list(csv.DictReader(f))
        print(f"\n{len(rows)} questions asked:", dict(Counter(r["status"] for r in rows)))
        for day, n in sorted(Counter(r["time"][:10] for r in rows).items())[-14:]:
            print(f"   {day}: {n}")